In [4]:
# Imports
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

import numpy as np
import pandas as pd

from xgboost import XGBClassifier


import warnings
warnings.filterwarnings('ignore')

In [5]:
# Load and merge data
trans = pd.read_csv('../data/raw/train_transaction.csv')
ident = pd.read_csv('../data/raw/train_identity.csv')

df = trans.merge(ident, on='TransactionID', how='left')
df = df.copy()

print(f"Shape: {df.shape}")
print("Data loaded successfully")

Shape: (590540, 434)
Data loaded successfully


In [6]:
# Time-based train/validation/test split
TRAIN_CUTOFF = 10_438_003
VAL_CUTOFF   = 13_151_880

df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)

train = df_sorted[df_sorted['TransactionDT'] < TRAIN_CUTOFF].copy()
val   = df_sorted[(df_sorted['TransactionDT'] >= TRAIN_CUTOFF) & 
                  (df_sorted['TransactionDT'] < VAL_CUTOFF)].copy()
test  = df_sorted[df_sorted['TransactionDT'] >= VAL_CUTOFF].copy()

print(f"Train: {len(train):,} rows | fraud rate: {train['isFraud'].mean():.4f}")
print(f"Val:   {len(val):,} rows | fraud rate: {val['isFraud'].mean():.4f}")
print(f"Test:  {len(test):,} rows | fraud rate: {test['isFraud'].mean():.4f}")
print(f"Total: {len(train)+len(val)+len(test):,} rows")

# Separate features and target
X_train = train.drop(columns=['isFraud', 'TransactionID'])
y_train = train['isFraud']

X_val = val.drop(columns=['isFraud', 'TransactionID'])
y_val = val['isFraud']

X_test = test.drop(columns=['isFraud', 'TransactionID'])
y_test = test['isFraud']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

Train: 413,378 rows | fraud rate: 0.0352
Val:   88,581 rows | fraud rate: 0.0343
Test:  88,581 rows | fraud rate: 0.0348
Total: 590,540 rows

X_train shape: (413378, 432)
X_val shape:   (88581, 432)
X_test shape:  (88581, 432)


In [7]:
# Datetime transformer
class DatetimeFeatures(BaseEstimator, TransformerMixin):
    """
    Extracts temporal features from TransactionDT.
    TransactionDT is a seconds offset from an unknown reference point.
    No fitting required, pure arithmetic transformation.
    """
    
    def fit(self, X, y=None):
        return self  # nothing to learn
    
    def transform(self, X):
        X = X.copy()
        X['hour_of_day']      = (X['TransactionDT'] // 3600) % 24
        # day_of_week: 7-day cycle derived from TransactionDT offset
        # Day 0 does not correspond to a known weekday, relative cycle only
        X['day_of_week']      = (X['TransactionDT'] // 86400) % 7
        X['is_weekend']       = (X['day_of_week'] >= 5).astype(int)
        X['time_since_start'] = X['TransactionDT']
        return X

# Test it
dt = DatetimeFeatures()
X_train_test = dt.fit_transform(X_train)

print("New columns added:")
print(X_train_test[['hour_of_day','day_of_week','is_weekend','time_since_start']].head())
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

New columns added:
   hour_of_day  day_of_week  is_weekend  time_since_start
0            0            1           0             86400
1            0            1           0             86401
2            0            1           0             86469
3            0            1           0             86499
4            0            1           0             86506

Shape before: (413378, 432) | after: (413378, 436)


In [8]:
# Cell 5 — Amount features transformer
class AmountFeatures(BaseEstimator, TransformerMixin):
    """
    Derives features from TransactionAmt.
    No fitting required — pure arithmetic transformation.
    Log transform handles right skew (skewness=14.37 found in EDA).
    """

    def fit(self, X, y=None):
        return self  # nothing to learn

    def transform(self, X):
        X = X.copy()
        X['amt_log']     = np.log1p(X['TransactionAmt'])
        X['amt_rounded'] = (X['TransactionAmt'] % 1 == 0).astype(int)
        X['amt_cents']   = X['TransactionAmt'] % 1
        return X

# Test it
amt = AmountFeatures()
X_train_test = amt.fit_transform(X_train)

print("New columns added:")
print(X_train_test[['TransactionAmt','amt_log','amt_rounded','amt_cents']].head())
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

New columns added:
   TransactionAmt   amt_log  amt_rounded  amt_cents
0            68.5  4.241327            0        0.5
1            29.0  3.401197            1        0.0
2            59.0  4.094345            1        0.0
3            50.0  3.931826            1        0.0
4            50.0  3.931826            1        0.0

Shape before: (413378, 432) | after: (413378, 435)


In [9]:
from sklearn.base import BaseEstimator, TransformerMixin

# Module-level constant — fixed domain-to-provider mapping
PROVIDER_MAP = {
    'gmail':      'google',
    'googlemail': 'google',
    'yahoo':      'yahoo',
    'ymail':      'yahoo',
    'rocketmail': 'yahoo',
    'hotmail':    'microsoft',
    'outlook':    'microsoft',
    'msn':        'microsoft',
    'live':       'microsoft',
    'att':        'att',
    'sbcglobal':  'att',
    'bellsouth':  'att',
    'icloud':     'apple',
    'me':         'apple',
    'mac':        'apple',
    'comcast':    'comcast',
    'aol':        'aol',
    'anonymous':  'anonymous',
    'missing':    'missing',   # ✅ Fix: preserve missing category
}


class EmailFeatures(BaseEstimator, TransformerMixin):
    """
    Email feature engineering for fraud detection.

    Features created:
    - P_email_freq: frequency of payer email domain
    - R_email_freq: frequency of receiver email domain
    - email_match: whether P and R domains are identical
    - P_email_provider: grouped provider (google, yahoo, etc.)

    Design principles:
    - No data leakage (fit only learns from training data)
    - Missing values handled explicitly as 'missing'
    - Unknown providers mapped to 'other'
    - Stable behavior across train / validation / test
    """

    def fit(self, X, y=None):
        X = X.copy()

        p_filled = X['P_emaildomain'].fillna('missing')
        r_filled = X['R_emaildomain'].fillna('missing')

        # Learn frequency distributions
        self.p_email_freq_ = p_filled.value_counts(normalize=True)
        self.r_email_freq_ = r_filled.value_counts(normalize=True)

        return self

    def transform(self, X):
        X = X.copy()

        # Fill missing consistently
        p_filled = X['P_emaildomain'].fillna('missing')
        r_filled = X['R_emaildomain'].fillna('missing')

        # Frequency encoding
        X['P_email_freq'] = p_filled.map(self.p_email_freq_).fillna(0)
        X['R_email_freq'] = r_filled.map(self.r_email_freq_).fillna(0)

        # Matching feature
        X['email_match'] = (p_filled == r_filled).astype(int)

        # Extract provider base safely
        p_provider_raw = p_filled.str.split('.', n=1).str[0]

        # Map to provider groups
        X['P_email_provider'] = (
            p_provider_raw
            .map(PROVIDER_MAP)
            .fillna('other')   # unknown providers → 'other'
        )

        return X
# Initialize
email = EmailFeatures()

# Fit
email.fit(X_train)

# Transform
X_train_transformed = email.transform(X_train)

# Quick checks
print(X_train_transformed[
    ['P_emaildomain', 'P_email_provider', 'P_email_freq', 'email_match']
].head(10))

print("\nProvider distribution:")
print(X_train_transformed['P_email_provider'].value_counts())

print("\nDomain → Provider mapping check:")
print(
    X_train_transformed[['P_emaildomain', 'P_email_provider']]
    .value_counts()
    .head(15)
)

   P_emaildomain P_email_provider  P_email_freq  email_match
0            NaN          missing      0.155538            1
1      gmail.com           google      0.385154            0
2    outlook.com        microsoft      0.008588            0
3      yahoo.com            yahoo      0.170314            0
4      gmail.com           google      0.385154            0
5      gmail.com           google      0.385154            0
6      yahoo.com            yahoo      0.170314            0
7       mail.com            other      0.000963            0
8  anonymous.com        anonymous      0.064699            0
9      yahoo.com            yahoo      0.170314            0

Provider distribution:
P_email_provider
google       159567
yahoo         74006
missing       64296
microsoft     42925
anonymous     26745
aol           19798
other          8233
att            6256
apple          5816
comcast        5736
Name: count, dtype: int64

Domain → Provider mapping check:
P_emaildomain  P_email_provi

In [10]:
# Cell 7 — Card aggregation features transformer
class CardAggFeatures(BaseEstimator, TransformerMixin):
    """
    Per-card behavioral aggregations using standard mean/std/count.
    fit() computes statistics from training data only.
    transform() maps any split using learned statistics.
    Unseen cards fall back to global training statistics.

    Known limitation: within-training leakage exists because each
    transaction contributes to its own card's mean. Impact is negligible
    at this dataset scale (~30 transactions per card on average).
    LOO encoding is documented as a future improvement.
    """

    def fit(self, X, y=None):
        stats = X.groupby('card1')['TransactionAmt'].agg(['mean','std','count'])
        self.card1_mean_   = stats['mean']
        self.card1_std_    = stats['std']
        self.card1_count_  = stats['count']
        self.global_mean_  = X['TransactionAmt'].mean()
        self.global_std_   = X['TransactionAmt'].std()
        return self

    def transform(self, X):
        X = X.copy()
        X['card1_amt_mean'] = X['card1'].map(self.card1_mean_).fillna(self.global_mean_)
        X['card1_amt_std']  = X['card1'].map(self.card1_std_).fillna(self.global_std_)
        X['card1_tx_count'] = X['card1'].map(self.card1_count_).fillna(1.0)
        return X

# Test on both train and val
card_agg = CardAggFeatures()
card_agg.fit(X_train)
X_train_test = card_agg.transform(X_train)
X_val_test   = card_agg.transform(X_val)

print("New columns added:")
print(X_train_test[['card1','card1_amt_mean','card1_amt_std','card1_tx_count']].head(8))
print(f"\nTrain shape: {X_train.shape} → {X_train_test.shape}")
print(f"Val shape:   {X_val.shape} → {X_val_test.shape}")

New columns added:
   card1  card1_amt_mean  card1_amt_std  card1_tx_count
0  13926      367.688929     383.506436              28
1   2755      244.516341     501.969085             522
2   4663       96.776658     102.847409             769
3  18132      122.976838     187.744406            2944
4   4497      105.083333      55.003125               9
5   5937      148.250000     101.257963               6
6  12308      107.889438     196.564359             160
7  12695      141.476273     206.269243            4772

Train shape: (413378, 432) → (413378, 435)
Val shape:   (88581, 432) → (88581, 435)


In [11]:
# Cell 8 — M features transformer (your version, corrected)
class MFeatures(BaseEstimator, TransformerMixin):
    """
    Encodes M1-M9 match flag features.
    M1-M3, M5-M9: three-state encoding T=1, F=0, NaN=-1
    M4: one-hot encoding with fixed columns learned during fit()
        Unseen categories in val/test dropped.
        Missing expected columns filled with 0.
    Adds M_sum_true and M_sum_false as behavioral aggregates.
    """

    def fit(self, X, y=None):
        self.binary_m = ['M1','M2','M3','M5','M6','M7','M8','M9']
        cats = set(X['M4'].dropna().unique())
        cats.add('missing')
        self.m4_categories_ = sorted([f"M4_{cat}" for cat in cats])
        return self

    def transform(self, X):
        X = X.copy()

        # Vectorized three-state encoding
        X[self.binary_m] = (
            X[self.binary_m]
            .replace({'T': 1, 'F': 0})
            .fillna(-1)
        )

        # One-hot encoding for M4 — enforce training columns
        m4_dummies = pd.get_dummies(
            X['M4'].fillna('missing'), prefix='M4'
        ).astype(int)

        m4_dummies = m4_dummies.reindex(
            columns=self.m4_categories_,
            fill_value=0
        )

        # Behavioral aggregates
        X['M_sum_true']  = (X[self.binary_m] == 1).sum(axis=1)
        X['M_sum_false'] = (X[self.binary_m] == 0).sum(axis=1)

        X = pd.concat([X.drop(columns=['M4']), m4_dummies], axis=1)
        return X

# Test on both train and val
m_feat = MFeatures()
m_feat.fit(X_train)
X_train_test = m_feat.transform(X_train)
X_val_test   = m_feat.transform(X_val)

print("M4 columns learned during fit:")
print(m_feat.m4_categories_)
print(X_train_test[['M1','M2','M3','M_sum_true','M_sum_false'] + m_feat.m4_categories_].head(8))
print(f"\nTrain shape: {X_train.shape} → {X_train_test.shape}")
print(f"Val shape:   {X_val.shape} → {X_val_test.shape}")

M4 columns learned during fit:
['M4_M0', 'M4_M1', 'M4_M2', 'M4_missing']
   M1  M2  M3  M_sum_true  M_sum_false  M4_M0  M4_M1  M4_M2  M4_missing
0   1   1   1           4            1      0      0      1           0
1  -1  -1  -1           2            0      1      0      0           0
2   1   1   1           3            5      1      0      0           0
3  -1  -1  -1           1            1      1      0      0           0
4  -1  -1  -1           0            0      0      0      0           1
5   1   1   1           4            1      0      1      0           0
6   1   1   1           6            2      1      0      0           0
7  -1  -1  -1           0            2      1      0      0           0

Train shape: (413378, 432) → (413378, 437)
Val shape:   (88581, 432) → (88581, 437)


In [12]:
class DFeatures(BaseEstimator, TransformerMixin):
    """
    Handles D1-D15 time delta features.
    Missing values imputed with -1 (sentinel — impossible natural value).
    Binary is_missing flag added per column.
    No fitting required.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        d_cols = [f'D{i}' for i in range(1, 16)]
        for col in d_cols:
            X[f'{col}_missing'] = X[col].isna().astype(int)
            X[col] = X[col].fillna(-1)
        return X

# Test it
d_feat = DFeatures()
X_train_test = d_feat.fit_transform(X_train)

print("D1 sample:")
print(X_train_test[['D1', 'D1_missing', 'D2', 'D2_missing']].head(8))
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")


D1 sample:
      D1  D1_missing     D2  D2_missing
0   14.0           0   -1.0           1
1    0.0           0   -1.0           1
2    0.0           0   -1.0           1
3  112.0           0  112.0           0
4    0.0           0   -1.0           1
5    0.0           0   -1.0           1
6    0.0           0   -1.0           1
7    0.0           0   -1.0           1

Shape before: (413378, 432) | after: (413378, 447)


In [13]:
class IdentityFeatures(BaseEstimator, TransformerMixin):
    """
    Handles identity features id_01 to id_38.
    75.6% of transactions have no identity data.
    Single has_identity flag added — 1 if any identity data present, 0 if not.
    All NaN values imputed with -999 (sentinel value).
    No fitting required.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        id_cols = [f'id_{str(i).zfill(2)}' for i in range(1, 39)]
        X['has_identity'] = X[id_cols].notna().any(axis=1).astype(int)
        for col in id_cols:
            X[col] = X[col].fillna(-999)
        return X
# Test it
id_feat = IdentityFeatures()
id_feat.fit(X_train)
X_train_test = id_feat.transform(X_train)

print("Identity features sample:")
print(X_train_test[['id_01', 'id_02', 'has_identity']].head(8))
print(f"\nhas_identity distribution:")
print(X_train_test['has_identity'].value_counts(normalize=True).round(4))
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

Identity features sample:
   id_01    id_02  has_identity
0 -999.0   -999.0             0
1 -999.0   -999.0             0
2 -999.0   -999.0             0
3 -999.0   -999.0             0
4    0.0  70787.0             1
5 -999.0   -999.0             0
6 -999.0   -999.0             0
7 -999.0   -999.0             0

has_identity distribution:
has_identity
0    0.7341
1    0.2659
Name: proportion, dtype: float64

Shape before: (413378, 432) | after: (413378, 433)


In [14]:
# Define V feature groups by missing rate
v_cols = [f'V{i}' for i in range(1, 340)]
missing_v = X_train[v_cols].isna().mean().round(2)

v_groups = {}
for threshold in missing_v.unique():
    group = missing_v[missing_v == threshold].index.tolist()
    v_groups[threshold] = group

for threshold, cols in sorted(v_groups.items()):
    print(f"Missing {threshold:.0%}: {len(cols)} features — {cols[:3]}...")

Missing 0%: 86 features — ['V95', 'V96', 'V97']...
Missing 15%: 45 features — ['V12', 'V13', 'V14']...
Missing 17%: 20 features — ['V75', 'V76', 'V77']...
Missing 30%: 18 features — ['V35', 'V36', 'V37']...
Missing 55%: 11 features — ['V1', 'V2', 'V3']...
Missing 74%: 66 features — ['V167', 'V168', 'V169']...
Missing 76%: 46 features — ['V217', 'V218', 'V219']...
Missing 84%: 47 features — ['V138', 'V139', 'V140']...


In [15]:
def validate_columns(X, columns, name):
    missing = [col for col in columns if col not in X.columns]

    if missing:
        raise ValueError(
            f"{name}: input is missing {len(missing)} expected columns: "
            f"{missing[:10]}"
        )

    return X.loc[:, columns]

In [16]:
class VRawImputedRepresentation(BaseEstimator, TransformerMixin):
    """
    RAW V representation.

    This is not native raw NaN handling.
    It intentionally uses median imputation + missing indicators so RAW and PCA
    carry the same missingness information.
    """

    def fit(self, X, y=None):
        self.columns_ = list(X.columns)

        X_checked = validate_columns(X, self.columns_, "VRawImputedRepresentation")

        self.imputer_ = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )
        self.imputer_.fit(X_checked)

        self.feature_names_out_ = (
            [f"{col}_value" for col in self.columns_]
            + [f"{col}_missing" for col in self.columns_]
        )

        return self

    def transform(self, X):
        X_checked = validate_columns(X, self.columns_, "VRawImputedRepresentation")

        values = self.imputer_.transform(X_checked)
        missing = X_checked.isna().astype("int8").values

        output = np.hstack([values, missing])

        return pd.DataFrame(
            output,
            columns=self.feature_names_out_,
            index=X_checked.index,
        )

    def get_feature_names_out(self):
        return np.array(self.feature_names_out_)

In [17]:
class VPCARepresentation(BaseEstimator, TransformerMixin):
    """
    PCA V representation.

    Design:
    - missing indicators are created before imputation
    - median imputation exists only because PCA cannot accept NaNs
    - scaling is used only before PCA, because PCA is scale-sensitive
    - no post-PCA scaling
    - output remains a DataFrame with feature names
    """

    def __init__(self, n_components=0.95, random_state=42):
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y=None):
        self.columns_ = list(X.columns)

        X_checked = validate_columns(X, self.columns_, "VPCARepresentation")

        self.imputer_ = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )
        self.scaler_ = StandardScaler()
        self.pca_ = PCA(
            n_components=self.n_components,
            random_state=self.random_state,
        )

        X_imputed = self.imputer_.fit_transform(X_checked)
        X_scaled = self.scaler_.fit_transform(X_imputed)

        self.pca_.fit(X_scaled)

        self.n_pca_components_ = self.pca_.n_components_

        self.feature_names_out_ = (
            [f"v_pca_{i+1:03d}" for i in range(self.n_pca_components_)]
            + [f"{col}_missing" for col in self.columns_]
        )

        return self

    def transform(self, X):
        X_checked = validate_columns(X, self.columns_, "VPCARepresentation")

        missing = X_checked.isna().astype("int8").values

        X_imputed = self.imputer_.transform(X_checked)
        X_scaled = self.scaler_.transform(X_imputed)
        X_pca = self.pca_.transform(X_scaled)

        output = np.hstack([X_pca, missing])

        return pd.DataFrame(
            output,
            columns=self.feature_names_out_,
            index=X_checked.index,
        )

    def get_feature_names_out(self):
        return np.array(self.feature_names_out_)

In [18]:
def make_xgb_benchmark_model(y_train):
    n_negative = (y_train == 0).sum()
    n_positive = (y_train == 1).sum()

    if n_positive == 0:
        raise ValueError("y_train contains no positive fraud cases.")

    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=n_negative / n_positive,
        random_state=42,
        n_jobs=-1,
    )

In [19]:
def evaluate_v_group(X_train, y_train, X_val, y_val, cols, representation):
    if representation == "raw":
        transformer = VRawImputedRepresentation()
    elif representation == "pca":
        transformer = VPCARepresentation(n_components=0.95, random_state=42)
    else:
        raise ValueError(f"Unknown representation: {representation}")

    pipe = Pipeline([
        ("v_representation", transformer),
        ("model", make_xgb_benchmark_model(y_train)),
    ])

    pipe.fit(X_train[cols], y_train)

    val_scores = pipe.predict_proba(X_val[cols])[:, 1]
    val_auc_pr = average_precision_score(y_val, val_scores)

    n_output_features = len(
        pipe.named_steps["v_representation"].get_feature_names_out()
    )

    return {
        "representation": representation,
        "n_input_features": len(cols),
        "n_output_features": n_output_features,
        "val_auc_pr": float(val_auc_pr),
    }

In [20]:
HIGH_MISSING_PCA_CUTOFF = 0.70

v_benchmark_results = []

for missing_rate, cols in sorted(v_groups.items()):
    print(f"\nV group missing={missing_rate:.0%} | {len(cols)} features")

    representations = ["raw"]

    if missing_rate <= HIGH_MISSING_PCA_CUTOFF:
        representations.append("pca")
    else:
        print("  PCA skipped: missing rate is too high")

    for representation in representations:
        print(f"  benchmarking {representation}")

        result = evaluate_v_group(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            cols=cols,
            representation=representation,
        )

        result["missing_rate"] = missing_rate
        v_benchmark_results.append(result)

v_benchmark_df = pd.DataFrame(v_benchmark_results)

v_benchmark_df = v_benchmark_df[
    [
        "missing_rate",
        "representation",
        "n_input_features",
        "n_output_features",
        "val_auc_pr",
    ]
]

v_benchmark_df.sort_values(
    ["missing_rate", "val_auc_pr"],
    ascending=[True, False],
)


V group missing=0% | 86 features
  benchmarking raw
  benchmarking pca

V group missing=15% | 45 features
  benchmarking raw
  benchmarking pca

V group missing=17% | 20 features
  benchmarking raw
  benchmarking pca

V group missing=30% | 18 features
  benchmarking raw
  benchmarking pca

V group missing=55% | 11 features
  benchmarking raw
  benchmarking pca

V group missing=74% | 66 features
  PCA skipped: missing rate is too high
  benchmarking raw

V group missing=76% | 46 features
  PCA skipped: missing rate is too high
  benchmarking raw

V group missing=84% | 47 features
  PCA skipped: missing rate is too high
  benchmarking raw


,missing_rate,representation,n_input_features,n_output_features,val_auc_pr
0,0.00,raw,86,172,0.218009
1,0.00,pca,86,118,0.208390
2,0.15,raw,45,90,0.212578
3,0.15,pca,45,62,0.202552
5,0.17,pca,20,29,0.211652
4,0.17,raw,20,40,0.208844
6,0.30,raw,18,36,0.233336
7,0.30,pca,18,27,0.220975
9,0.55,pca,11,18,0.068568
8,0.55,raw,11,22,0.065770


In [21]:
MIN_AUC_PR_GAIN_FOR_PCA = 0.001
MIN_FEATURE_REDUCTION_FOR_PCA_TIE = 0.50

v_strategy = {}

for missing_rate, group_df in v_benchmark_df.groupby("missing_rate"):
    raw_row = group_df[group_df["representation"] == "raw"].iloc[0]
    pca_rows = group_df[group_df["representation"] == "pca"]

    if len(pca_rows) == 0:
        selected = raw_row
        reason = "raw selected because PCA was skipped for high missingness"
    else:
        pca_row = pca_rows.iloc[0]

        pca_gain = pca_row["val_auc_pr"] - raw_row["val_auc_pr"]
        feature_reduction = 1 - (
            pca_row["n_output_features"] / raw_row["n_output_features"]
        )

        if pca_gain >= MIN_AUC_PR_GAIN_FOR_PCA:
            selected = pca_row
            reason = "pca selected because it improved validation AUC-PR"
        elif (
            pca_gain >= -MIN_AUC_PR_GAIN_FOR_PCA
            and feature_reduction >= MIN_FEATURE_REDUCTION_FOR_PCA_TIE
        ):
            selected = pca_row
            reason = "pca selected because AUC-PR was similar with major compression"
        else:
            selected = raw_row
            reason = "raw selected because PCA did not justify itself"

    v_strategy[missing_rate] = {
        "strategy": selected["representation"],
        "reason": reason,
        "n_input_features": int(selected["n_input_features"]),
        "n_output_features": int(selected["n_output_features"]),
        "val_auc_pr": float(selected["val_auc_pr"]),
    }

v_strategy

{0.0: {'strategy': 'raw',
  'reason': 'raw selected because PCA did not justify itself',
  'n_input_features': 86,
  'n_output_features': 172,
  'val_auc_pr': 0.21800912968622124},
 0.15: {'strategy': 'raw',
  'reason': 'raw selected because PCA did not justify itself',
  'n_input_features': 45,
  'n_output_features': 90,
  'val_auc_pr': 0.2125775039093961},
 0.17: {'strategy': 'pca',
  'reason': 'pca selected because it improved validation AUC-PR',
  'n_input_features': 20,
  'n_output_features': 29,
  'val_auc_pr': 0.21165174797814224},
 0.3: {'strategy': 'raw',
  'reason': 'raw selected because PCA did not justify itself',
  'n_input_features': 18,
  'n_output_features': 36,
  'val_auc_pr': 0.23333644218412886},
 0.55: {'strategy': 'pca',
  'reason': 'pca selected because it improved validation AUC-PR',
  'n_input_features': 11,
  'n_output_features': 18,
  'val_auc_pr': 0.06856758916909406},
 0.74: {'strategy': 'raw',
  'reason': 'raw selected because PCA was skipped for high missi

V-feature benchmark decision:

This benchmark compares two representations for each V missing-rate group using
the same model family as the final system: XGBoost.

RAW is represented as median-imputed V values plus explicit missing indicators.
PCA is represented as scaled PCA components plus the same missing indicators.

PCA is skipped for groups with missing rate above 70%, because in very sparse
groups the imputation pattern can dominate the PCA structure.

The selected V strategy is still a local representation choice. It must be
validated again inside the full feature pipeline with amount, card, email,
M, D, and identity features.

V-feature strategy — locked after benchmark (02_feature_engineering.ipynb)
Groups 0.17 and 0.55 → PCA (improved AUC-PR)
Groups 0.0, 0.15, 0.30, 0.84 → Raw (PCA did not justify itself)
Group 0.84 → PCA skipped (missing rate > 70%)
Validation pending: strategy re-evaluated inside full pipeline (Step 4)

In [22]:
v_benchmark_df.sort_values(["missing_rate", "val_auc_pr"], ascending=[True, False])

,missing_rate,representation,n_input_features,n_output_features,val_auc_pr
0,0.00,raw,86,172,0.218009
1,0.00,pca,86,118,0.208390
2,0.15,raw,45,90,0.212578
3,0.15,pca,45,62,0.202552
5,0.17,pca,20,29,0.211652
4,0.17,raw,20,40,0.208844
6,0.30,raw,18,36,0.233336
7,0.30,pca,18,27,0.220975
9,0.55,pca,11,18,0.068568
8,0.55,raw,11,22,0.065770


The 55% missing group selects PCA by validation AUC-PR, but both raw and PCA
perform weakly in isolation. This choice should be revalidated in the full
pipeline before finalizing.


In [23]:
import re
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

RAW_V_DERIVED_PATTERN = re.compile(r"^V\d+($|_.*)")


def require_columns(X, columns, owner):
    missing = [col for col in columns if col not in X.columns]
    if missing:
        raise ValueError(
            f"{owner}: missing {len(missing)} required columns. "
            f"First missing: {missing[:10]}"
        )
    return X.loc[:, columns]


def assert_unique_columns(columns, owner):
    duplicated = pd.Index(columns)[pd.Index(columns).duplicated()].tolist()
    if duplicated:
        raise ValueError(f"{owner}: duplicate columns found: {duplicated[:10]}")


def assert_no_unprefixed_v_derivatives(columns, owner):
    bad = [col for col in columns if RAW_V_DERIVED_PATTERN.match(str(col))]
    if bad:
        raise ValueError(
            f"{owner}: unprefixed raw V-derived columns leaked: {bad[:10]}"
        )


In [24]:
class StableCategoricalEncoder(BaseEstimator, TransformerMixin):
    """
    Stable non-target categorical encoder.

    Low-cardinality:
    - fixed train-learned categories
    - unseen values mapped to __OTHER__
    - missing values mapped to __MISSING__
    - feature names are stored during fit, never inferred from empty frames

    High-cardinality:
    - smoothed frequency encoding
    - missing is encoded as the frequency of __MISSING__
    - unseen values get a learned fallback frequency
    """

    def __init__(self, max_onehot_cardinality=20, alpha=1.0):
        self.max_onehot_cardinality = max_onehot_cardinality
        self.alpha = alpha
        self.missing_token = "__MISSING__"
        self.other_token = "__OTHER__"

    def _clean_values(self, series):
        return series.astype("string").fillna(self.missing_token).astype(str)

    def fit(self, X, y=None):
        self.input_columns_ = list(X.columns)
        assert_unique_columns(self.input_columns_, "StableCategoricalEncoder.fit")

        self.categorical_cols_ = X.select_dtypes(
            include=["object", "category", "string"]
        ).columns.tolist()

        self.numeric_cols_ = [
            col for col in self.input_columns_
            if col not in self.categorical_cols_
        ]

        self.onehot_categories_ = {}
        self.onehot_feature_names_ = {}
        self.onehot_index_ = {}
        self.onehot_lookup_ = {}

        self.freq_maps_ = {}
        self.unseen_freq_ = {}

        for col in self.categorical_cols_:
            values = self._clean_values(X[col])
            counts = values.value_counts(dropna=False)

            if len(counts) <= self.max_onehot_cardinality:
                categories = counts.index.tolist()

                for token in [self.missing_token, self.other_token]:
                    if token not in categories:
                        categories.append(token)

                feature_names = [
                    f"{col}__onehot_{i:03d}"
                    for i in range(len(categories))
                ]

                self.onehot_categories_[col] = categories
                self.onehot_feature_names_[col] = feature_names
                self.onehot_index_[col] = {
                    category: i for i, category in enumerate(categories)
                }
                self.onehot_lookup_[col] = dict(zip(feature_names, categories))

            else:
                denom = len(values) + self.alpha * (len(counts) + 1)
                self.freq_maps_[col] = (counts + self.alpha) / denom
                self.unseen_freq_[col] = self.alpha / denom

        self.feature_names_out_ = list(self.numeric_cols_)

        for col in self.categorical_cols_:
            if col in self.onehot_feature_names_:
                self.feature_names_out_.extend(self.onehot_feature_names_[col])
            else:
                self.feature_names_out_.append(f"{col}__freq")

        assert_unique_columns(
            self.feature_names_out_,
            "StableCategoricalEncoder.feature_names_out_",
        )

        return self

    def transform(self, X):
        X = require_columns(X, self.input_columns_, "StableCategoricalEncoder")
        parts = [X.loc[:, self.numeric_cols_].copy()]

        for col in self.categorical_cols_:
            values = self._clean_values(X[col])

            if col in self.onehot_categories_:
                categories = self.onehot_categories_[col]
                category_set = set(categories) - {self.other_token}

                values = values.where(values.isin(category_set), self.other_token)
                codes = values.map(self.onehot_index_[col]).astype("int64").to_numpy()

                arr = np.zeros((len(X), len(categories)), dtype="int8")
                arr[np.arange(len(X)), codes] = 1

                parts.append(
                    pd.DataFrame(
                        arr,
                        columns=self.onehot_feature_names_[col],
                        index=X.index,
                    )
                )

            else:
                encoded = values.map(self.freq_maps_[col]).fillna(
                    self.unseen_freq_[col]
                )

                parts.append(
                    pd.DataFrame(
                        {f"{col}__freq": encoded.astype("float32")},
                        index=X.index,
                    )
                )

        output = pd.concat(parts, axis=1)
        return output.loc[:, self.feature_names_out_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_out_, dtype=object)


In [25]:
class NumericFinalizer(BaseEstimator, TransformerMixin):
    """
    Final numeric safety layer.

    No new missing indicators are created here.
    All intentional missingness features must be created upstream.
    """

    def fit(self, X, y=None):
        assert_unique_columns(X.columns, "NumericFinalizer.fit")

        non_numeric = [
            col for col in X.columns
            if not pd.api.types.is_numeric_dtype(X[col])
        ]

        if non_numeric:
            raise TypeError(f"Non-numeric columns remain: {non_numeric[:20]}")

        self.columns_ = list(X.columns)
        clean = X.replace([np.inf, -np.inf], np.nan)
        self.medians_ = clean.median(numeric_only=True).fillna(0)

        return self

    def transform(self, X):
        X = require_columns(X, self.columns_, "NumericFinalizer").copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians_)
        return X.loc[:, self.columns_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columns_, dtype=object)


In [26]:
class FraudFeaturePipeline(BaseEstimator, TransformerMixin):
    """
    Strict full feature pipeline.

    Main guarantees:
    - fit only learns from train
    - transform requires the training schema
    - feature block inputs are checked before each block
    - unprefixed raw V-derived columns are blocked
    - final output is numeric, named, stable, and NaN-free
    """

    def __init__(self, v_groups, v_strategy, max_onehot_cardinality=20):
        self.v_groups = v_groups
        self.v_strategy = v_strategy
        self.max_onehot_cardinality = max_onehot_cardinality

    def _lookup_v_strategy(self, missing_rate):
        if missing_rate in self.v_strategy:
            return self.v_strategy[missing_rate]["strategy"]

        for key, value in self.v_strategy.items():
            if np.isclose(float(key), float(missing_rate)):
                return value["strategy"]

        raise ValueError(f"No V strategy found for missing rate {missing_rate}")

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("FraudFeaturePipeline expects a pandas DataFrame.")

        assert_unique_columns(X.columns, "FraudFeaturePipeline.fit")

        self.input_columns_ = list(X.columns)
        X_raw = X.copy()

        self.v_source_cols_ = sorted(
            {col for cols in self.v_groups.values() for col in cols}
        )

        self.datetime_ = DatetimeFeatures().fit(X_raw, y)
        self.amount_ = AmountFeatures().fit(X_raw, y)
        self.email_ = EmailFeatures().fit(X_raw, y)
        self.card_ = CardAggFeatures().fit(X_raw, y)
        self.m_ = MFeatures().fit(X_raw, y)
        self.d_ = DFeatures().fit(X_raw, y)
        self.identity_ = IdentityFeatures().fit(X_raw, y)

        self.feature_blocks_ = [
            ("datetime", self.datetime_, ["TransactionDT"]),
            ("amount", self.amount_, ["TransactionAmt"]),
            ("email", self.email_, ["P_emaildomain", "R_emaildomain"]),
            ("card", self.card_, ["card1", "TransactionAmt"]),
            ("m", self.m_, [f"M{i}" for i in range(1, 10)]),
            ("d", self.d_, [f"D{i}" for i in range(1, 16)]),
            (
                "identity",
                self.identity_,
                [f"id_{i:02d}" for i in range(1, 39)],
            ),
        ]

        self.v_representations_ = []

        for missing_rate, cols in sorted(self.v_groups.items()):
            strategy = self._lookup_v_strategy(missing_rate)

            if strategy == "raw":
                transformer = VRawImputedRepresentation()
            elif strategy == "pca":
                transformer = VPCARepresentation(n_components=0.95, random_state=42)
            else:
                raise ValueError(f"Unknown V strategy: {strategy}")

            source = require_columns(
                X_raw,
                cols,
                f"V group missing={missing_rate:.2f}",
            )

            transformer.fit(source, y)

            self.v_representations_.append(
                (missing_rate, list(cols), strategy, transformer)
            )

        engineered = self._engineer_features(X_raw)

        self.categorical_ = StableCategoricalEncoder(
            max_onehot_cardinality=self.max_onehot_cardinality
        ).fit(engineered, y)

        encoded = self.categorical_.transform(engineered)

        self.finalizer_ = NumericFinalizer().fit(encoded, y)
        self.feature_names_out_ = self.finalizer_.get_feature_names_out()

        return self

    def transform(self, X):
        X_raw = require_columns(
            X,
            self.input_columns_,
            "FraudFeaturePipeline.transform",
        ).copy()

        engineered = self._engineer_features(X_raw)
        encoded = self.categorical_.transform(engineered)
        finalized = self.finalizer_.transform(encoded)

        if finalized.isna().any().any():
            raise ValueError("Final feature table still contains NaNs.")

        assert_unique_columns(finalized.columns, "FraudFeaturePipeline.transform")

        return finalized

    def _apply_feature_blocks(self, X):
        X_work = X.copy()

        for name, transformer, required_cols in self.feature_blocks_:
            require_columns(X_work, required_cols, f"{name} block input")
            X_work = transformer.transform(X_work)

            if not isinstance(X_work, pd.DataFrame):
                raise TypeError(f"{name} block did not return a DataFrame.")

            assert_unique_columns(X_work.columns, f"{name} block output")

        return X_work

    def _engineer_features(self, X):
        X_original = X.copy()
        X_work = self._apply_feature_blocks(X_original)

        main = X_work.drop(columns=self.v_source_cols_, errors="ignore")
        main = main.drop(columns=["TransactionID", "isFraud"], errors="ignore")

        assert_no_unprefixed_v_derivatives(
            main.columns,
            "main feature block",
        )

        v_parts = []

        for missing_rate, cols, strategy, transformer in self.v_representations_:
            group_name = f"vgrp_{int(round(float(missing_rate) * 100)):02d}"

            source = require_columns(
                X_original,
                cols,
                f"V group missing={missing_rate:.2f}",
            )

            part = transformer.transform(source)
            part = part.add_prefix(f"{group_name}__")
            v_parts.append(part)

        output = pd.concat([main] + v_parts, axis=1)

        assert_unique_columns(output.columns, "engineered feature output")
        assert_no_unprefixed_v_derivatives(
            [col for col in output.columns if not str(col).startswith("vgrp_")],
            "engineered feature output",
        )

        return output

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_out_, dtype=object)


In [27]:
feature_pipeline = FraudFeaturePipeline(
    v_groups=v_groups,
    v_strategy=v_strategy,
    max_onehot_cardinality=20,
)

feature_pipeline.fit(X_train, y_train)

X_train_processed = feature_pipeline.transform(X_train)
X_val_processed = feature_pipeline.transform(X_val)
X_test_processed = feature_pipeline.transform(X_test)

print("Train:", X_train_processed.shape)
print("Val:  ", X_val_processed.shape)
print("Test: ", X_test_processed.shape)
print("NaNs:", X_train_processed.isna().sum().sum())
print("Non-numeric:", X_train_processed.select_dtypes(exclude="number").shape[1])
print("Duplicate columns:", X_train_processed.columns.duplicated().sum())

Train: (413378, 905)
Val:   (88581, 905)
Test:  (88581, 905)
NaNs: 0
Non-numeric: 0
Duplicate columns: 0


In [28]:
import sys
sys.path.insert(0, "/Users/ilyas/Desktop/ieee-fraud-detection")
print(sys.path[0])

/Users/ilyas/Desktop/ieee-fraud-detection


In [29]:
from src.features.build_features import FraudFeaturePipeline

feature_pipeline = FraudFeaturePipeline(
    v_groups=v_groups,
    v_strategy=v_strategy,
    max_onehot_cardinality=20,
)

feature_pipeline.fit(X_train, y_train)

X_train_processed = feature_pipeline.transform(X_train)
X_val_processed = feature_pipeline.transform(X_val)
X_test_processed = feature_pipeline.transform(X_test)

print("Train:", X_train_processed.shape)
print("Val:  ", X_val_processed.shape)
print("Test: ", X_test_processed.shape)
print("NaNs:", X_train_processed.isna().sum().sum())
print("Non-numeric:", X_train_processed.select_dtypes(exclude="number").shape[1])
print("Duplicate columns:", X_train_processed.columns.duplicated().sum())


Train: (413378, 905)
Val:   (88581, 905)
Test:  (88581, 905)
NaNs: 0
Non-numeric: 0
Duplicate columns: 0


In [30]:
from pathlib import Path
import joblib

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_train_processed.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_val_processed.to_csv(PROCESSED_DIR / "X_val.csv", index=False)
X_test_processed.to_csv(PROCESSED_DIR / "X_test.csv", index=False)

y_train.to_frame("isFraud").to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_val.to_frame("isFraud").to_csv(PROCESSED_DIR / "y_val.csv", index=False)
y_test.to_frame("isFraud").to_csv(PROCESSED_DIR / "y_test.csv", index=False)

joblib.dump(feature_pipeline, PROCESSED_DIR / "feature_pipeline.joblib")

print("Saved processed data to:")
print(PROCESSED_DIR.resolve())


Saved processed data to:
/Users/ilyas/Desktop/ieee-fraud-detection/data/processed
